In [ ]:
from typing import Any

from pytorch_lightning.utilities.types import STEP_OUTPUT

""" Class 25 | Project 2 | Machine Translation using Pretrained Model

Objectives:
1. End-to-end machine translation training pipeline
2. Fine-tune a pre-trained model for the custom dataset
"""

import pytorch_lightning as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from torchmetrics.text import BLEUScore
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [ ]:
"""Task: English to Bangla """

mt_pretrained_model_name = "shhossain/opus-mt-en-to-bn"

In [ ]:
""" For NLP tasks, we basically need two entities:
1. Tokenizer
2. Model
"""

tokenizer = AutoTokenizer.from_pretrained(mt_pretrained_model_name)
mt_pretrained_model = AutoModelForSeq2SeqLM.from_pretrained(mt_pretrained_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/969k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


model.safetensors:   0%|          | 0.00/303M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

# Data

In [ ]:
"""
Sentence: How are you, dude?
Tokens: 'How', 'are', 'you', 'dude?'
ids: 125, 14, 145, 78
max_length = 3
ids: [125, 14, 145]
"""

class MTDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = str(self.data.iloc[idx]['en'])
        tgt_text = str(self.data.iloc[idx]['bn'])

        src_encoding = tokenizer(
            src_text,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        tgt_encoding = tokenizer(
            tgt_text,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'src_input_ids': src_encoding['input_ids'].squeeze(),
            'src_attention_mask': src_encoding['attention_mask'].squeeze(),
            'tgt_input_ids': tgt_encoding['input_ids'].squeeze(),
            'tgt_attention_mask': tgt_encoding['attention_mask'].squeeze()
        }

"""
example: How are you, dude?
input_ids: 125, 14, 145, 78
max_length = 7
input_ids: [125, 14, 145, 147, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 0, 0, 0]
"""

'\nexample: How are you, dude?\ninput_ids: 125, 14, 145, 78\nmax_length = 7\ninput_ids: [125, 14, 145, 147, 0, 0, 0]\nattention_mask: [1, 1, 1, 1, 0, 0, 0]\n'

In [ ]:
class MTDataModule(pl.LightningDataModule):
    def __init__(self, train_csv, val_csv, test_csv, batch_size=32):
        super().__init__()
        self.train_csv = train_csv
        self.val_csv = val_csv
        self.test_csv = test_csv
        self.batch_size = batch_size

    def setup(self, stage=None):
        self.train_dataset = MTDataset(self.train_csv)
        self.val_dataset = MTDataset(self.val_csv)
        self.test_dataset = MTDataset(self.test_csv)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

In [ ]:
data_module = MTDataModule(
    train_csv='/content/train.csv',
    val_csv='/content/val.csv',
    test_csv='/content/test.csv',
    batch_size=32
)

# Model

In [ ]:
class MTModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        # load pretrained model
        self.model = AutoModelForSeq2SeqLM.from_pretrained(mt_pretrained_model_name)
        # load pretrained tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(mt_pretrained_model_name)
        # learning rate
        self.learning_rate = 2e-5
        # loss function
        self.loss_fn = nn.CrossEntropyLoss(
            ignore_index=self.tokenizer.pad_token_id
        )
        # evaluation metric
        self.bleu = BLEUScore()

    def forward(self,
                src_input_ids,
                src_attention_mask,
                tgt_input_ids,
                tgt_attention_mask
        ):
        outputs = self.model(
            input_ids=src_input_ids,
            attention_mask=src_attention_mask,
            decoder_input_ids=tgt_input_ids[:, :-1],
            decoder_attention_mask=tgt_attention_mask[:, :-1]
        )
        return outputs

    def training_step(self, batch, batch_idx):
        loss = self.compute_loss(batch, batch_idx, 'train')
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self.compute_loss(batch, batch_idx, 'val')
        self.log('val_loss', loss, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        loss = self.compute_loss(batch, batch_idx, 'test')
        self.log('test_loss', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=10
        )
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}

    def compute_loss(self, batch, batch_idx, stage):
        src_input_ids = batch['src_input_ids']
        src_attention_mask = batch['src_attention_mask']
        tgt_input_ids = batch['tgt_input_ids']
        tgt_attention_mask = batch['tgt_attention_mask']

        outputs = self(
            src_input_ids,
            src_attention_mask,
            tgt_input_ids,
            tgt_attention_mask
        )
        logits = outputs.logits
        loss = self.loss_fn(
            logits.view(-1, logits.size(-1)),
            tgt_input_ids[:, 1:].contiguous().view(-1)
        )

        if stage == 'val' or stage == 'test':
            preds = torch.argmax(logits, dim=-1)
            pred_texts = self.tokenizer.batch_decode(preds, skip_special_tokens=True)
            tgt_texts = self.tokenizer.batch_decode(tgt_input_ids[:, 1:], skip_special_tokens=True)
            bleu_score = self.bleu(pred_texts, [[tgt] for tgt in tgt_texts])
            self.log(f'{stage}_bleu', bleu_score, prog_bar=True)

        return loss


In [ ]:
model = MTModel()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
trainer = pl.Trainer(
    max_epochs=5,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    precision=16,
    log_every_n_steps=10,
    val_check_interval=0.25
)

/usr/local/lib/python3.12/dist-packages/lightning_fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model, data_module)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ MarianMTModel    │ 76.3 M │ eval  │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
│ 2 │ bleu    │ BLEUScore        │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 76.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 76.3 M                                                                                               
Total estimated model params size (MB): 305                                                                        
Modules in train mode: 2                                                                                           
Modules in eval mode: 180                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 180 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.